In [4]:
import numpy as np    # for interim processing for the ECG Signal 
import pywt # for wavelet decomposition
import neurokit2 as nk  # to leverage the functions of neurokit2 to identify, the different segments of sample 

# We're getting serious now with the below mentioned - Advanced Signal Processing
from scipy.signal import butter, \
                        lfilter, \
                        savgol_filter       

def run_pan_tompkins_segmentation(signal,  # the ECG signal 
                             sampling_rate=300.0  # for the scope of this project, it would stay at 300 Hz but other datasets on physionet might have different freq
                             ):
    
    """The famous algorithm to detect the QRS interval from 1985 by Jiapu Pan(Shanghai) and Willis Tompkins(USA)
       running on 8-Bit Z80 processor of the time. Today, we have Floating proint processing units that can easily 
        hammer down the floating point values from the signal"""

# Nyquist frequency represents the theoretical maximum bandwidth limit (Fs / 2)
    # At 300 Hz sampling rate, the Nyquist frequency is exactly 150 Hz
    nyq = 0.5 * sampling_rate           
    
    # Normalize our 5 Hz and 15 Hz cutoff frequencies relative to the Nyquist limit
    low, high = 5.0 / nyq, 15.0 / nyq           
    
    # Design and apply a 2nd-order cascaded Butterworth Band-pass filter (5-15 Hz)
    # This isolates the QRS energy while attenuating muscle noise, 60Hz hum, and T-wave interference
    b, a = butter(1, [low, high], btype='band') 
    filtered_signal = lfilter(b, a, signal)     
    
    # Five-point first-derivative approximation in the discrete-time domain
    # This highlights the steep slopes/gradients of the R-wave upstroke
    derivative = np.diff(filtered_signal)       
    
    # Point-by-point squaring operation to make all data points non-negative 
    # and non-linearly magnify high-frequency QRS complexes over lower-frequency components
    squared = derivative ** 2                   
    
    # Define a 150ms moving window width (45 samples at 300 Hz) to match standard QRS duration
    window_width = int(0.15 * sampling_rate)    
    
    # Discrete-time linear convolution implementing a Moving Window Integration (MWI) 
    # to obtain a smooth, moving energy envelope of the QRS complex
    integrated = np.convolve(squared, np.ones(window_width)/window_width, mode='same')  
    
    # Calculate a dynamic signal-to-noise threshold profile baseline
    threshold = np.mean(integrated) * 1.8       
    r_peaks = []
    threshold_breaches = np.where(integrated > threshold)[0]
    
    # Group continuous blocks of true indices that breached the envelope threshold
    if len(threshold_breaches) > 0:
        groups = np.split(threshold_breaches, np.where(np.diff(threshold_breaches) > 1)[0] + 1)
        for group in groups:
            if len(group) == 0: continue
            # Find the absolute maximum amplitude point within the raw signal window
            actual_signal_window = signal[group]
            r_peaks.append(int(group[np.argmax(actual_signal_window)]))
            
    # Apply a 400ms physiological refractory period blanking window 
    # to eliminate false positive multiple-triggers on the same cardiac contraction
    r_peaks = [p for i, p in enumerate(r_peaks) if i == 0 or (p - r_peaks[i-1]) > int(sampling_rate * 0.4)]

    # Relative localized physiological search windows for sub-wave segmentation
    p_points, q_points, s_points, t_points = [], [], [], []
    for r in r_peaks:
        if r < 120 or r > len(signal) - 150: continue
        
        # Q-dip: Local minimum immediately preceding the R-peak
        q_idx = (r - 12) + np.argmin(signal[r-12:r])
        q_points.append(int(q_idx))
        
        # S-dip: Local minimum immediately following the R-peak
        s_idx = r + np.argmin(signal[r:r+12])
        s_points.append(int(s_idx))
        
        # P-wave: Local maximum corresponding to atrial depolarization
        p_idx = (r - 60) + np.argmax(signal[r-60:r-15])
        p_points.append(int(p_idx))
        
        # T-wave: Local maximum corresponding to ventricular repolarization
        t_idx = (r + 30) + np.argmax(signal[r+30:r+120])
        t_points.append(int(t_idx))

    return {"P": p_points, "Q": q_points, "R": r_peaks, "S": s_points, "T": t_points}


In [5]:
import os # for file IO and stuff related to dataset
import matplotlib.pyplot as plt # for plotting  
import pandas as pd # out goto guy for processing CSVs and many more things
from IPython.display import Image, display  # Added for viewing in notebook
import numpy as np # for running interim data conversions
import wfdb  # for reading the ECG samples

In [6]:
annotations_processed_csv =  pd.read_csv("C:\Project\Research-Project\DataFrame_Extracts\processed_annotations.csv")
annotations_processed_csv.head()

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units
0,A00001,N,C:\Project\Research-Project\training2017\\A00001,300,9000,1,mV
1,A00002,N,C:\Project\Research-Project\training2017\\A00002,300,9000,1,mV
2,A00003,N,C:\Project\Research-Project\training2017\\A00003,300,18000,1,mV
3,A00004,A,C:\Project\Research-Project\training2017\\A00004,300,9000,1,mV
4,A00005,A,C:\Project\Research-Project\training2017\\A00005,300,18000,1,mV


In [7]:
# selecting all samples with more then 9k records
annotations_processed_9K_samples = annotations_processed_csv.loc[annotations_processed_csv.Signal_Length_Samples>=9000 ]

In [8]:
annotations_processed_9K_samples

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units
0,A00001,N,C:\Project\Research-Project\training2017\\A00001,300,9000,1,mV
1,A00002,N,C:\Project\Research-Project\training2017\\A00002,300,9000,1,mV
2,A00003,N,C:\Project\Research-Project\training2017\\A00003,300,18000,1,mV
3,A00004,A,C:\Project\Research-Project\training2017\\A00004,300,9000,1,mV
4,A00005,A,C:\Project\Research-Project\training2017\\A00005,300,18000,1,mV
...,...,...,...,...,...,...,...
8523,A08524,N,C:\Project\Research-Project\training2017\\A08524,300,9000,1,mV
8524,A08525,O,C:\Project\Research-Project\training2017\\A08525,300,9000,1,mV
8525,A08526,N,C:\Project\Research-Project\training2017\\A08526,300,10904,1,mV
8526,A08527,N,C:\Project\Research-Project\training2017\\A08527,300,9000,1,mV


In [9]:
from sklearn.model_selection import StratifiedShuffleSplit
# we're using StratifiedShuffleSplit to balance the classes, if we use a regular train-test split the resultant split might be a tad unbalanced 

In [10]:
#this function takes care of the balanced split 
def split_dataset_stratified(df, train_ratio=0.8, random_state=42):
    """
    Function 1 (Corrected): A regular, stratified train-test split.
    Because every record is still a single row at this point, this balances
    the medical classes perfectly across Train and Test without any leakage.
    """
    # Initialize the stratified splitter for a single split
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=train_ratio, random_state=random_state)
    
    # Split using the 'Inference' column to keep class distributions identical
    for train_idx, test_idx in splitter.split(df, df['Inference']):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
    
    return train_df, test_df

In [11]:
# this is just the raw validation split
raw_train, raw_test = split_dataset_stratified(annotations_processed_9K_samples, train_ratio=0.8)

This is the piece-de-resistance of this run, splitting long records to make more records for trainning

In [12]:
def slice_ecg_records(df, set_name="Dataset"):
    """
    Function 2: Slices records based on lengths and logs detailed dataset statistics,
    including dropped/split counts and class size transformations.
    """
    sliced_rows = []
    
    # Trackers for logging
    total_input = len(df)
    dropped_count = 0
    single_chunk_count = 0
    split_count = 0
    
    print(f"\n======== LOGGING STATISTICS FOR: {set_name.upper()} ========")
    print(f"Initial total records: {total_input}")
    
    # Capture initial class distribution (Inference column)
    initial_classes = df['Inference'].value_counts()
    
    for _, row in df.iterrows():
        pid = row['SampleID']
        length = int(row['Signal_Length_Samples'])
        
        # Rule 1: Less than 9k -> Ignore
        if length < 9000:
            dropped_count += 1
            continue
            
        # Rule 3: 13.5k to 18k -> Split into 2 samples (Head & Tail)
        if length >= 13500:
            split_count += 1
            
            # Record 1: Head window
            s1 = row.to_dict()
            s1['Sliced_SampleID'] = f"{pid}_1"
            s1['Start_Idx'] = 0
            s1['End_Idx'] = 9000
            sliced_rows.append(s1)
            
            # Record 2: Tail window
            s2 = row.to_dict()
            s2['Sliced_SampleID'] = f"{pid}_2"
            s2['Start_Idx'] = length - 9000
            s2['End_Idx'] = length
            sliced_rows.append(s2)
            
        # Rule 2: 9k to 13.5k -> Take first 9k points
        else:
            single_chunk_count += 1
            s_fixed = row.to_dict()
            s_fixed['Sliced_SampleID'] = f"{pid}_1"
            s_fixed['Start_Idx'] = 0
            s_fixed['End_Idx'] = 9000
            sliced_rows.append(s_fixed)
            
    # Convert to DataFrame to calculate final stats
    sliced_df = pd.DataFrame(sliced_rows)
    total_output = len(sliced_df) if not sliced_df.empty else 0
    
    # Print processing logs
    print(f"Dropped records (< 9k): {dropped_count}")
    print(f"Standard records (9k - 13.5k): {single_chunk_count}")
    print(f"Split records (>= 13.5k): {split_count} (Generated {split_count * 2} samples)")
    print(f"Final sliced total records: {total_output}")
    print("-" * 45)
    
    # Print class transformation logs
    print("CLASS SIZE TRANSFORMATION (Initial -> Sliced):")
    if not sliced_df.empty:
        final_classes = sliced_df['Inference'].value_counts()
        # Combine initial and final for a side-by-side view
        all_classes = sorted(list(set(initial_classes.index) | set(final_classes.index)))
        for cls in all_classes:
            init_val = initial_classes.get(cls, 0)
            final_val = final_classes.get(cls, 0)
            print(f"  * Class [{cls}]: {init_val} records -> {final_val} samples")
    else:
        print("  No samples generated.")
    print("=" * 45)
    
    return sliced_df


In [13]:
grown_train_set = slice_ecg_records(raw_train, set_name="Train Set")
grown_test_set = slice_ecg_records(raw_test, set_name="Test Set")


======== LOGGING STATISTICS FOR: TRAIN SET ========
Initial total records: 6048
Dropped records (< 9k): 0
Standard records (9k - 13.5k): 5226
Split records (>= 13.5k): 822 (Generated 1644 samples)
Final sliced total records: 6870
---------------------------------------------
CLASS SIZE TRANSFORMATION (Initial -> Sliced):
  * Class [A]: 500 records -> 580 samples
  * Class [N]: 3623 records -> 4026 samples
  * Class [O]: 1809 records -> 2138 samples
  * Class [~]: 116 records -> 126 samples

======== LOGGING STATISTICS FOR: TEST SET ========
Initial total records: 1513
Dropped records (< 9k): 0
Standard records (9k - 13.5k): 1295
Split records (>= 13.5k): 218 (Generated 436 samples)
Final sliced total records: 1731
---------------------------------------------
CLASS SIZE TRANSFORMATION (Initial -> Sliced):
  * Class [A]: 125 records -> 147 samples
  * Class [N]: 906 records -> 1017 samples
  * Class [O]: 453 records -> 538 samples
  * Class [~]: 29 records -> 29 samples


In [14]:
# Earlier we had this but since we've grown the dataset, we have to manually assign the X_train, y_train, X_test, y_test datasets
#X = Sample_wtih_9k_points.drop(columns=["Inference"])
#Y = Sample_wtih_9k_points["Inference"]


X_train = grown_train_set.drop(columns=["Inference"])
y_train = grown_train_set["Inference"]
X_test  = grown_test_set.drop(columns=["Inference"])
y_test  = grown_test_set["Inference"]
print(f" X_train shape: {X_train.shape} , X_test shape:{X_test.shape}, \n  y_train shape:{y_train.shape}, y_test shape:{y_test.shape}")

 X_train shape: (6870, 9) , X_test shape:(1731, 9), 
  y_train shape:(6870,), y_test shape:(1731,)


In [ ]:
grown_train_set

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units,Sliced_SampleID,Start_Idx,End_Idx
0,A03095,O,C:\Project\Research-Project\training2017\\A03095,300,9000,1,mV,A03095_1,0,9000
1,A03688,N,C:\Project\Research-Project\training2017\\A03688,300,13398,1,mV,A03688_1,0,9000
2,A07992,O,C:\Project\Research-Project\training2017\\A07992,300,9000,1,mV,A07992_1,0,9000
3,A06423,A,C:\Project\Research-Project\training2017\\A06423,300,9000,1,mV,A06423_1,0,9000
4,A07892,O,C:\Project\Research-Project\training2017\\A07892,300,9000,1,mV,A07892_1,0,9000
...,...,...,...,...,...,...,...,...,...,...
6865,A05311,O,C:\Project\Research-Project\training2017\\A05311,300,9000,1,mV,A05311_1,0,9000
6866,A03944,O,C:\Project\Research-Project\training2017\\A03944,300,9000,1,mV,A03944_1,0,9000
6867,A02698,N,C:\Project\Research-Project\training2017\\A02698,300,9000,1,mV,A02698_1,0,9000
6868,A02977,N,C:\Project\Research-Project\training2017\\A02977,300,9000,1,mV,A02977_1,0,9000


In [38]:
grown_train_set.loc[grown_train_set.Signal_Length_Samples>13500]

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units,Sliced_SampleID,Start_Idx,End_Idx
24,A04313,N,C:\Project\Research-Project\training2017\\A04313,300,18000,1,mV,A04313_1,0,9000
25,A04313,N,C:\Project\Research-Project\training2017\\A04313,300,18000,1,mV,A04313_2,9000,18000
30,A06910,N,C:\Project\Research-Project\training2017\\A06910,300,18000,1,mV,A06910_1,0,9000
31,A06910,N,C:\Project\Research-Project\training2017\\A06910,300,18000,1,mV,A06910_2,9000,18000
37,A00817,O,C:\Project\Research-Project\training2017\\A00817,300,18000,1,mV,A00817_1,0,9000
...,...,...,...,...,...,...,...,...,...,...
6849,A04880,N,C:\Project\Research-Project\training2017\\A04880,300,18000,1,mV,A04880_2,9000,18000
6855,A05290,N,C:\Project\Research-Project\training2017\\A05290,300,18000,1,mV,A05290_1,0,9000
6856,A05290,N,C:\Project\Research-Project\training2017\\A05290,300,18000,1,mV,A05290_2,9000,18000
6863,A06850,O,C:\Project\Research-Project\training2017\\A06850,300,18000,1,mV,A06850_1,0,9000


In [52]:
grown_train_set.loc[grown_train_set.Start_Idx != 0]

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units,Sliced_SampleID,Start_Idx,End_Idx
25,A04313,N,C:\Project\Research-Project\training2017\\A04313,300,18000,1,mV,A04313_2,9000,18000
31,A06910,N,C:\Project\Research-Project\training2017\\A06910,300,18000,1,mV,A06910_2,9000,18000
38,A00817,O,C:\Project\Research-Project\training2017\\A00817,300,18000,1,mV,A00817_2,9000,18000
40,A07103,N,C:\Project\Research-Project\training2017\\A07103,300,18000,1,mV,A07103_2,9000,18000
43,A07632,O,C:\Project\Research-Project\training2017\\A07632,300,18000,1,mV,A07632_2,9000,18000
...,...,...,...,...,...,...,...,...,...,...
6843,A03568,N,C:\Project\Research-Project\training2017\\A03568,300,18000,1,mV,A03568_2,9000,18000
6845,A01854,N,C:\Project\Research-Project\training2017\\A01854,300,18000,1,mV,A01854_2,9000,18000
6849,A04880,N,C:\Project\Research-Project\training2017\\A04880,300,18000,1,mV,A04880_2,9000,18000
6856,A05290,N,C:\Project\Research-Project\training2017\\A05290,300,18000,1,mV,A05290_2,9000,18000


In [47]:
#dataframe Columns:
#SampleID	Matlab_files	Sampling_Rate_Hz	Signal_Length_Samples	Num_Channels	Units	Sliced_SampleID	Start_Idx	End_Idx

current_Rec = grown_train_set.iloc[0]
raw_path = current_Rec["Matlab_files"] 
print(raw_path)

C:\Project\Research-Project\training2017\\A03095


In [76]:
# 1. Extract the raw string path from the dataframe
raw_path = grown_train_set["Matlab_files"].iloc[0]

# 2. Strip the extension (.mat or .hea) if your column includes it
ecg_record_name = os.path.splitext(raw_path)[0]

# 3. Read the recording correctly
signals, fields = wfdb.rdsamp(ecg_record_name)

# 4. extract the points from the base signal to reconstruct the current sample slice   
rec_ECG_Sample = signals[current_Rec["Start_Idx"]:current_Rec.End_Idx, 0]

In [78]:
pqrst_dict = pqrst_dict = run_pan_tompkins_segmentation(rec_ECG_Sample,  # the ECG signal 
                             sampling_rate=300.0  # for the scope of this project, it would stay at 300 Hz but other datasets on physionet might have different freq
                             )
pqrst_dict

{'P': [369,
  927,
  1443,
  1766,
  2211,
  2720,
  3038,
  3458,
  3856,
  4174,
  4490,
  4989,
  5335,
  5813,
  6160,
  6529,
  6906,
  7304,
  7604,
  8042,
  8351],
 'Q': [414,
  931,
  1458,
  1793,
  2255,
  2748,
  3047,
  3498,
  3889,
  4186,
  4501,
  4993,
  5376,
  5842,
  6177,
  6573,
  6910,
  7316,
  7632,
  8067,
  8367],
 'R': [424,
  943,
  1470,
  1804,
  2267,
  2760,
  3059,
  3510,
  3900,
  4198,
  4510,
  5005,
  5388,
  5854,
  6187,
  6585,
  6922,
  7328,
  7644,
  8076,
  8379,
  8866],
 'S': [433,
  951,
  1478,
  1812,
  2275,
  2769,
  3066,
  3517,
  3908,
  4206,
  4517,
  5012,
  5397,
  5863,
  6194,
  6593,
  6929,
  7335,
  7652,
  8084,
  8386],
 'T': [531,
  1041,
  1575,
  1905,
  2361,
  2858,
  3147,
  3590,
  3988,
  4294,
  4612,
  5104,
  5475,
  5949,
  6288,
  6685,
  7004,
  7427,
  7730,
  8168,
  8479]}

In [74]:
import json
import os
 

# 1. Define a helper function to process a single row
def extract_pqrst_for_row(row):
    # Get the base record path (removing .mat extension if present)
    record_path = os.path.splitext(row["Matlab_files"])[0] 
    print(f"record_path: {record_path}")
    try:
        # Read the signal data
        signal, fields = wfdb.rdsamp(record_path)

        signal = signal[row["Start_Idx"]:row.End_Idx, 0]
        #print(f"signal: {signal}")

        pqrst_dict = run_pan_tompkins_segmentation(signal,  # the ECG signal 
                             sampling_rate=300.0  # for the scope of this project, it would stay at 300 Hz but other datasets on physionet might have different freq
                             )
        print(f"pqrst_dict: {pqrst_dict}")
 
        # Return as a JSON string (or just return the dict if you want a live object)
        return json.dumps(pqrst_dict)

    except Exception as e:
        # Handle missing files or extraction errors gracefully
        return json.dumps({"error": str(e)})


 

In [75]:
A = extract_pqrst_for_row(current_Rec)

record_path: C:\Project\Research-Project\training2017\\A03095
pqrst_dict: {'P': [369, 927, 1443, 1766, 2211, 2720, 3038, 3458, 3856, 4174, 4490, 4989, 5335, 5813, 6160, 6529, 6906, 7304, 7604, 8042, 8351], 'Q': [414, 931, 1458, 1793, 2255, 2748, 3047, 3498, 3889, 4186, 4501, 4993, 5376, 5842, 6177, 6573, 6910, 7316, 7632, 8067, 8367], 'R': [424, 943, 1470, 1804, 2267, 2760, 3059, 3510, 3900, 4198, 4510, 5005, 5388, 5854, 6187, 6585, 6922, 7328, 7644, 8076, 8379, 8866], 'S': [433, 951, 1478, 1812, 2275, 2769, 3066, 3517, 3908, 4206, 4517, 5012, 5397, 5863, 6194, 6593, 6929, 7335, 7652, 8084, 8386], 'T': [531, 1041, 1575, 1905, 2361, 2858, 3147, 3590, 3988, 4294, 4612, 5104, 5475, 5949, 6288, 6685, 7004, 7427, 7730, 8168, 8479]}


In [61]:
A

'{"error": "object too deep for desired array"}'

In [79]:
# 2. Apply the function to the entire DataFrame
X_train["PQRST_Points"] = X_train.apply(extract_pqrst_for_row, axis=1)


record_path: C:\Project\Research-Project\training2017\\A03095
pqrst_dict: {'P': [369, 927, 1443, 1766, 2211, 2720, 3038, 3458, 3856, 4174, 4490, 4989, 5335, 5813, 6160, 6529, 6906, 7304, 7604, 8042, 8351], 'Q': [414, 931, 1458, 1793, 2255, 2748, 3047, 3498, 3889, 4186, 4501, 4993, 5376, 5842, 6177, 6573, 6910, 7316, 7632, 8067, 8367], 'R': [424, 943, 1470, 1804, 2267, 2760, 3059, 3510, 3900, 4198, 4510, 5005, 5388, 5854, 6187, 6585, 6922, 7328, 7644, 8076, 8379, 8866], 'S': [433, 951, 1478, 1812, 2275, 2769, 3066, 3517, 3908, 4206, 4517, 5012, 5397, 5863, 6194, 6593, 6929, 7335, 7652, 8084, 8386], 'T': [531, 1041, 1575, 1905, 2361, 2858, 3147, 3590, 3988, 4294, 4612, 5104, 5475, 5949, 6288, 6685, 7004, 7427, 7730, 8168, 8479]}
record_path: C:\Project\Research-Project\training2017\\A03688
pqrst_dict: {'P': [263, 500, 733, 978, 1223, 1463, 1700, 1945, 2189, 2427, 2676, 2929, 3152, 3420, 3653, 3894, 4103, 4319, 4571, 4769, 5005, 5262, 5522, 5781, 6044, 6307, 6550, 6796, 7032, 7267, 7499, 

In [80]:
X_train.to_csv(r"C://Project//Research-Project//DataFrame_Extracts//X_train.csv")